In [1]:
# ============================================================
# 10B_AURORA_component_ablation_standalone.ipynb
# Single-cell standalone version
#
# Purpose:
# - Reload ETF return panel
# - Reload Notebook 07 purged walk-forward probability files
# - Reconstruct AURORA10-UAMV-B allocation logic
# - Run seven controlled ablations
# - Export S22 ablation results and S23 cash/turnover diagnostics
#
# Research backtest only.
# Not personalized financial advice.
# ============================================================

from __future__ import annotations

import json
import math
import copy
import hashlib
import warnings
from pathlib import Path
from datetime import datetime, timezone

warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("Google Drive mount skipped or failed.")

import numpy as np
import pandas as pd

try:
    from scipy.optimize import minimize
    HAS_SCIPY = True
except Exception:
    HAS_SCIPY = False
    print("scipy.optimize not available. Optimizer will use fallback allocations.")


# ============================================================
# 1. Paths and global settings
# ============================================================

PROJECT_CODE = "AURORA_TWETF"
PUBLICATION_ROOT = Path("/content/drive/MyDrive/AURORA_TWETF")

DATA_ROOT = PUBLICATION_ROOT / "data"
PANEL_DIR = DATA_ROOT / "panels"
MODELING_DIR = DATA_ROOT / "modeling"

OUTPUT_ROOT = PUBLICATION_ROOT / "outputs" / PROJECT_CODE
TABLE_DIR = OUTPUT_ROOT / "tables"
REPORT_DIR = OUTPUT_ROOT / "reports"

NOTEBOOK07_RUN_ID = "20260624_031817"
NOTEBOOK07_ROOT = OUTPUT_ROOT / "purged_walk_forward_models" / f"run_{NOTEBOOK07_RUN_ID}"

NOTEBOOK08_INPUT_INDEX = NOTEBOOK07_ROOT / "NOTEBOOK08_OR_ALLOCATION_INPUT_INDEX_PURGED_WF.csv"

RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

RUN_ROOT = OUTPUT_ROOT / "aurora_component_ablation" / f"run_{RUN_ID}"

ABLATION_RETURN_DIR = RUN_ROOT / "returns"
ABLATION_WEIGHT_DIR = RUN_ROOT / "weights"
ABLATION_TABLE_DIR = RUN_ROOT / "tables"
ABLATION_DIAG_DIR = RUN_ROOT / "diagnostics"
ABLATION_REPORT_DIR = RUN_ROOT / "reports"

for d in [
    RUN_ROOT,
    ABLATION_RETURN_DIR,
    ABLATION_WEIGHT_DIR,
    ABLATION_TABLE_DIR,
    ABLATION_DIAG_DIR,
    ABLATION_REPORT_DIR,
    TABLE_DIR,
    REPORT_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

ETF_UNIVERSE = ["0050", "006208", "00692", "00881"]
CASH_COL = "CASH"
ALL_ASSETS = ETF_UNIVERSE + [CASH_COL]

CLASS_LABELS = [0, 1, 2, 3, 4]

ANNUALIZATION_DAYS = 252
INITIAL_CAPITAL = 1.0
TRANSACTION_COST_RATE = 0.0010
REBALANCE_FREQUENCY = "monthly"

STRICT_START = "2024-11-27"
STRICT_END = "2026-03-25"

FALLBACK_CONSTRAINED_EQUAL = {
    "0050": 0.30,
    "006208": 0.30,
    "00692": 0.25,
    "00881": 0.15,
    "CASH": 0.00,
}

FALLBACK_DEFENSIVE = {
    "0050": 0.24,
    "006208": 0.24,
    "00692": 0.20,
    "00881": 0.12,
    "CASH": 0.20,
}

BASE_UAMV_B_CONFIG = {
    "strategy_name": "UAMV_B_more60_defensive",
    "alpha_20d": 0.30,
    "alpha_60d": 0.70,
    "lookback_mu": 63,
    "lookback_cov": 126,
    "mean_shrinkage_to_zero": 0.60,
    "momentum_weight": 0.40,
    "base_risk_aversion": 10.0,
    "uncertainty_risk_multiplier": 2.5,
    "bearish_risk_multiplier": 2.0,
    "turnover_penalty": 0.25,
    "regime_tilt_strength": 0.25,
    "max_etf_weight": 0.45,
    "max_00881_weight": 0.30,
    "max_cash_weight": 0.60,
    "min_cash_weight": 0.00,
}

print("=" * 80)
print("AURORA-TWETF standalone component ablation")
print("=" * 80)
print("RUN_ID:", RUN_ID)
print("RUN_ROOT:", RUN_ROOT)
print("Notebook 07 input index:", NOTEBOOK08_INPUT_INDEX)
print("=" * 80)

if not NOTEBOOK08_INPUT_INDEX.exists():
    raise FileNotFoundError(f"Missing probability input index: {NOTEBOOK08_INPUT_INDEX}")


# ============================================================
# 2. Utility functions
# ============================================================

def save_json(path, obj):
    Path(path).write_text(
        json.dumps(obj, indent=2, ensure_ascii=False, default=str),
        encoding="utf-8",
    )

def clean_symbol_name(x):
    x = str(x)
    x = x.replace(".TW", "")
    x = x.replace(".TWO", "")
    x = x.replace("TW_", "")
    return x

def find_first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None

def safe_name(x):
    return (
        str(x)
        .replace("/", "_")
        .replace("\\", "_")
        .replace(":", "_")
        .replace(" ", "_")
        .replace(".", "_")
    )

def read_table_auto(path):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")

    if path.suffix.lower() == ".parquet":
        df = pd.read_parquet(path)
    elif path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
    else:
        raise ValueError(f"Unsupported file type: {path}")

    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"])
        df = df.set_index("date")
    else:
        df.index = pd.to_datetime(df.index)

    df.index.name = "date"
    return df.sort_index()

def load_etf_return_panel():
    candidates = [
        PANEL_DIR / "AURORA_etf_return_panel.parquet",
        PANEL_DIR / "AURORA_etf_returns_panel.parquet",
        PANEL_DIR / "AURORA_return_panel.parquet",
        MODELING_DIR / "AURORA_etf_return_panel.parquet",
    ]

    path = find_first_existing(candidates)

    if path is None:
        raise FileNotFoundError(
            "Could not find ETF return panel. Tried:\n"
            + "\n".join(str(p) for p in candidates)
        )

    df = pd.read_parquet(path)
    df.index = pd.to_datetime(df.index)
    df.index.name = "date"
    df = df.sort_index()
    df = df.rename(columns={c: clean_symbol_name(c) for c in df.columns})

    missing = [s for s in ETF_UNIVERSE if s not in df.columns]
    if missing:
        raise ValueError(
            f"ETF return panel found at {path}, but missing ETF columns: {missing}. "
            f"Available columns: {list(df.columns)}"
        )

    df = df[ETF_UNIVERSE].copy()
    df = df.replace([np.inf, -np.inf], np.nan).fillna(0.0)

    return df, path

def proba_cols():
    return [f"proba_class_{i}" for i in CLASS_LABELS]

def normalize_proba(p):
    p = np.asarray(p, dtype=float)
    p = np.nan_to_num(p, nan=0.0, posinf=0.0, neginf=0.0)
    p[p < 0] = 0.0

    row_sums = p.sum(axis=1, keepdims=True)
    zero_rows = row_sums[:, 0] <= 0

    if np.any(zero_rows):
        p[zero_rows, :] = 1.0 / p.shape[1]
        row_sums = p.sum(axis=1, keepdims=True)

    return p / row_sums

def normalize_vector(w):
    w = pd.Series(w, index=ALL_ASSETS, dtype=float)
    w = w.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    w[w < 0] = 0.0

    if w.sum() <= 0:
        w = pd.Series(FALLBACK_CONSTRAINED_EQUAL, dtype=float).reindex(ALL_ASSETS).fillna(0.0)

    w = w / w.sum()
    return w

def normalize_rows(df):
    out = df.copy()
    out = out.reindex(columns=ALL_ASSETS).fillna(0.0)
    out = out.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    out[out < 0] = 0.0

    row_sums = out.sum(axis=1)
    zero_mask = row_sums <= 0

    if zero_mask.any():
        out.loc[zero_mask, ETF_UNIVERSE] = 1.0 / len(ETF_UNIVERSE)
        out.loc[zero_mask, CASH_COL] = 0.0
        row_sums = out.sum(axis=1)

    out = out.div(row_sums, axis=0)
    return out

def latest_fold_deduplicate(proba_df, split_filter=None):
    df = proba_df.copy()

    if split_filter is not None and "split" in df.columns:
        if isinstance(split_filter, str):
            split_filter = [split_filter]
        df = df[df["split"].isin(split_filter)].copy()

    if df.empty:
        return df

    df = df.reset_index()

    if "fold_id" not in df.columns:
        df["fold_id"] = "WF0"

    df["fold_number"] = (
        df["fold_id"]
        .astype(str)
        .str.extract(r"(\d+)", expand=False)
        .fillna("0")
        .astype(int)
    )

    split_priority = {"train": 0, "validation": 1, "test": 2}
    if "split" in df.columns:
        df["split_priority"] = df["split"].map(split_priority).fillna(0).astype(int)
    else:
        df["split_priority"] = 0

    df = df.sort_values(["date", "split_priority", "fold_number"])
    df = df.drop_duplicates(subset=["date"], keep="last")
    df = df.set_index("date").sort_index()
    df.index.name = "date"

    return df.drop(columns=["fold_number", "split_priority"], errors="ignore")


# ============================================================
# 3. AURORA allocation functions
# ============================================================

def probability_features_from_blend(p20, p60, config):
    common = p20.index.intersection(p60.index).sort_values()

    p20 = p20.loc[common].copy()
    p60 = p60.loc[common].copy()

    prob20 = normalize_proba(p20[proba_cols()].values)
    prob60 = normalize_proba(p60[proba_cols()].values)

    alpha20 = float(config["alpha_20d"])
    alpha60 = float(config["alpha_60d"])

    if alpha20 + alpha60 <= 0:
        alpha20, alpha60 = 0.5, 0.5
    else:
        s = alpha20 + alpha60
        alpha20, alpha60 = alpha20 / s, alpha60 / s

    prob = alpha20 * prob20 + alpha60 * prob60
    prob = normalize_proba(prob)

    class_values = np.asarray(CLASS_LABELS, dtype=float)
    expected_class = prob @ class_values

    clipped = np.clip(prob, 1e-12, 1.0)
    entropy = -np.sum(clipped * np.log(clipped), axis=1)
    normalized_entropy = entropy / np.log(len(CLASS_LABELS))
    confidence = 1.0 - normalized_entropy

    sorted_p = np.sort(prob, axis=1)
    margin = sorted_p[:, -1] - sorted_p[:, -2]
    ordinal_variance = (prob @ (class_values ** 2)) - expected_class ** 2

    out = pd.DataFrame(index=common)
    out["expected_class"] = expected_class
    out["entropy"] = entropy
    out["normalized_entropy"] = normalized_entropy
    out["confidence_score"] = confidence
    out["probability_margin"] = margin
    out["ordinal_variance"] = ordinal_variance
    out["p_strong_bear"] = prob[:, 0]
    out["p_bear"] = prob[:, 1]
    out["p_neutral"] = prob[:, 2]
    out["p_bull"] = prob[:, 3]
    out["p_strong_bull"] = prob[:, 4]
    out["p_bearish"] = prob[:, 0] + prob[:, 1]
    out["p_bullish"] = prob[:, 3] + prob[:, 4]
    out["risk_on_score"] = (expected_class - 2.0) / 2.0

    for i, label in enumerate(CLASS_LABELS):
        out[f"proba_class_{label}"] = prob[:, i]

    return out

def apply_caps_to_vector(w, config):
    w = normalize_vector(w)

    max_etf_weight = float(config["max_etf_weight"])
    max_00881_weight = float(config["max_00881_weight"])
    max_cash_weight = float(config["max_cash_weight"])
    min_cash_weight = float(config["min_cash_weight"])

    w[CASH_COL] = min(max(w[CASH_COL], min_cash_weight), max_cash_weight)

    for etf in ETF_UNIVERSE:
        cap = max_etf_weight
        if etf == "00881":
            cap = min(cap, max_00881_weight)
        w[etf] = min(w[etf], cap)

    if w.sum() <= 0:
        w = pd.Series(FALLBACK_CONSTRAINED_EQUAL, dtype=float).reindex(ALL_ASSETS).fillna(0.0)

    w = w / w.sum()

    for _ in range(10):
        excess = 0.0
        capped = []

        for etf in ETF_UNIVERSE:
            cap = max_etf_weight
            if etf == "00881":
                cap = min(cap, max_00881_weight)

            if w[etf] > cap:
                excess += w[etf] - cap
                w[etf] = cap
                capped.append(etf)

        if w[CASH_COL] > max_cash_weight:
            excess += w[CASH_COL] - max_cash_weight
            w[CASH_COL] = max_cash_weight
            capped.append(CASH_COL)

        if w[CASH_COL] < min_cash_weight:
            needed = min_cash_weight - w[CASH_COL]
            w[CASH_COL] = min_cash_weight
            risky_sum = w[ETF_UNIVERSE].sum()
            if risky_sum > 0:
                w[ETF_UNIVERSE] *= max(0.0, risky_sum - needed) / risky_sum

        if excess <= 1e-12:
            break

        eligible = []
        for asset in ALL_ASSETS:
            if asset in capped:
                continue

            if asset == CASH_COL:
                if w[asset] < max_cash_weight:
                    eligible.append(asset)
            elif asset == "00881":
                if w[asset] < min(max_etf_weight, max_00881_weight):
                    eligible.append(asset)
            else:
                if w[asset] < max_etf_weight:
                    eligible.append(asset)

        if not eligible:
            break

        eligible_sum = w[eligible].sum()

        if eligible_sum <= 0:
            for asset in eligible:
                w[asset] += excess / len(eligible)
        else:
            w[eligible] += excess * (w[eligible] / eligible_sum)

        w = w.clip(lower=0.0)
        w = w / w.sum()

    return normalize_vector(w)

def fallback_weight(features_row, config):
    p_bearish = float(features_row.get("p_bearish", 0.0))
    confidence = float(features_row.get("confidence_score", 0.5))

    if p_bearish > 0.45 and confidence > 0.25:
        base = pd.Series(FALLBACK_DEFENSIVE, dtype=float)
    else:
        base = pd.Series(FALLBACK_CONSTRAINED_EQUAL, dtype=float)

    return apply_caps_to_vector(base.reindex(ALL_ASSETS).fillna(0.0), config)

def estimate_moments_for_date(etf_returns, date, features_row, config):
    lookback_mu = int(config["lookback_mu"])
    lookback_cov = int(config["lookback_cov"])

    hist_all = etf_returns.loc[etf_returns.index < date, ETF_UNIVERSE].copy()

    if len(hist_all) < max(30, min(lookback_mu, lookback_cov) // 2):
        return None, None

    hist_mu = hist_all.tail(lookback_mu)
    hist_cov = hist_all.tail(lookback_cov)

    if len(hist_mu) < 20 or len(hist_cov) < 30:
        return None, None

    mean_short = hist_mu.mean().values
    momentum_return = (1.0 + hist_mu).prod().values - 1.0
    momentum_daily = momentum_return / max(len(hist_mu), 1)

    momentum_weight = float(config["momentum_weight"])
    shrink = float(config["mean_shrinkage_to_zero"])

    mu_risky = (
        (1.0 - momentum_weight) * mean_short
        + momentum_weight * momentum_daily
    )

    mu_risky = (1.0 - shrink) * mu_risky

    risk_on = float(features_row.get("risk_on_score", 0.0))
    p_bearish = float(features_row.get("p_bearish", 0.0))
    p_bullish = float(features_row.get("p_bullish", 0.0))

    tilt_strength = float(config["regime_tilt_strength"])

    tilt = pd.Series(0.0, index=ETF_UNIVERSE)
    tilt["0050"] += 0.15 * risk_on
    tilt["006208"] += 0.15 * risk_on
    tilt["00692"] += 0.05 * risk_on
    tilt["00881"] += 0.35 * risk_on + 0.15 * p_bullish - 0.20 * p_bearish
    tilt["00692"] += 0.10 * p_bearish

    realized_vol = hist_cov.std().replace(0.0, np.nan)
    vol_scale = realized_vol.median()

    if not np.isfinite(vol_scale) or vol_scale <= 0:
        vol_scale = 0.01

    mu_risky = mu_risky + tilt_strength * tilt.values * vol_scale / ANNUALIZATION_DAYS

    cov_risky = hist_cov.cov().values
    cov_risky = np.nan_to_num(cov_risky, nan=0.0, posinf=0.0, neginf=0.0)

    avg_var = np.mean(np.diag(cov_risky))
    if not np.isfinite(avg_var) or avg_var <= 0:
        avg_var = 1e-4

    cov_risky = cov_risky + np.eye(len(ETF_UNIVERSE)) * avg_var * 0.10

    mu = np.zeros(len(ALL_ASSETS), dtype=float)
    mu[:len(ETF_UNIVERSE)] = mu_risky
    mu[-1] = 0.0

    cov = np.zeros((len(ALL_ASSETS), len(ALL_ASSETS)), dtype=float)
    cov[:len(ETF_UNIVERSE), :len(ETF_UNIVERSE)] = cov_risky
    cov[-1, -1] = 1e-10

    return mu, cov

def risk_aversion_for_date(features_row, config):
    base = float(config["base_risk_aversion"])
    uncertainty_mult = float(config["uncertainty_risk_multiplier"])
    bearish_mult = float(config["bearish_risk_multiplier"])

    uncertainty = float(features_row.get("normalized_entropy", 0.5))
    p_bearish = float(features_row.get("p_bearish", 0.0))

    lam = base * (1.0 + uncertainty_mult * uncertainty + bearish_mult * p_bearish)
    return float(max(lam, 1e-6))

def optimize_single_date(mu, cov, prev_w, features_row, config):
    if mu is None or cov is None or not HAS_SCIPY:
        return fallback_weight(features_row, config), "fallback_no_moments_or_scipy"

    max_etf_weight = float(config["max_etf_weight"])
    max_00881_weight = float(config["max_00881_weight"])
    max_cash_weight = float(config["max_cash_weight"])
    min_cash_weight = float(config["min_cash_weight"])
    turnover_penalty = float(config["turnover_penalty"])

    lam = risk_aversion_for_date(features_row, config)
    prev = normalize_vector(prev_w).values

    bounds = []
    for asset in ALL_ASSETS:
        if asset == CASH_COL:
            bounds.append((min_cash_weight, max_cash_weight))
        elif asset == "00881":
            bounds.append((0.0, min(max_etf_weight, max_00881_weight)))
        else:
            bounds.append((0.0, max_etf_weight))

    def objective(w):
        w = np.asarray(w, dtype=float)
        expected_return = float(mu @ w)
        variance = float(w.T @ cov @ w)
        turnover_term = float(np.sum((w - prev) ** 2))
        utility = expected_return - lam * variance - turnover_penalty * turnover_term
        return -utility

    constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]
    x0 = apply_caps_to_vector(prev, config).values

    try:
        res = minimize(
            objective,
            x0=x0,
            method="SLSQP",
            bounds=bounds,
            constraints=constraints,
            options={"maxiter": 300, "ftol": 1e-10, "disp": False},
        )

        if res.success and np.all(np.isfinite(res.x)):
            w = apply_caps_to_vector(res.x, config)
            return w, "optimized"

        w = fallback_weight(features_row, config)
        return w, f"fallback_optimizer_failed_{str(res.message)[:80]}"

    except Exception as e:
        w = fallback_weight(features_row, config)
        return w, f"fallback_exception_{repr(e)[:80]}"

def build_uamv_signal_weights(p20, p60, etf_returns, config, evaluation_dates):
    features = probability_features_from_blend(p20, p60, config)

    evaluation_dates = (
        pd.DatetimeIndex(evaluation_dates)
        .intersection(features.index)
        .intersection(etf_returns.index)
        .sort_values()
    )

    if len(evaluation_dates) == 0:
        raise ValueError(f"No evaluation dates for {config['strategy_name']}")

    rows = []
    diagnostics = []

    prev_w = pd.Series(FALLBACK_CONSTRAINED_EQUAL, dtype=float).reindex(ALL_ASSETS).fillna(0.0)
    prev_w = apply_caps_to_vector(prev_w, config)

    for dt in evaluation_dates:
        features_row = features.loc[dt]

        mu, cov = estimate_moments_for_date(
            etf_returns=etf_returns,
            date=dt,
            features_row=features_row,
            config=config,
        )

        w, status = optimize_single_date(
            mu=mu,
            cov=cov,
            prev_w=prev_w,
            features_row=features_row,
            config=config,
        )

        rows.append(w.values)

        diag = {
            "date": dt,
            "strategy_name": config["strategy_name"],
            "optimization_status": status,
            "risk_aversion": risk_aversion_for_date(features_row, config),
            "expected_class": float(features_row["expected_class"]),
            "p_bearish": float(features_row["p_bearish"]),
            "p_bullish": float(features_row["p_bullish"]),
            "normalized_entropy": float(features_row["normalized_entropy"]),
            "confidence_score": float(features_row["confidence_score"]),
            "ordinal_variance": float(features_row["ordinal_variance"]),
        }

        diagnostics.append(diag)
        prev_w = w

    weight_df = pd.DataFrame(rows, index=evaluation_dates, columns=ALL_ASSETS)
    weight_df.index.name = "date"
    weight_df = normalize_rows(weight_df)

    diagnostic_df = pd.DataFrame(diagnostics).set_index("date").sort_index()
    diagnostic_df.index.name = "date"

    return weight_df, diagnostic_df


# ============================================================
# 4. Backtest functions
# ============================================================

def get_rebalance_dates(index, frequency):
    idx = pd.DatetimeIndex(index).sort_values()

    if frequency == "monthly":
        groups = pd.Series(idx, index=idx).groupby([idx.year, idx.month])
    elif frequency == "quarterly":
        groups = pd.Series(idx, index=idx).groupby([idx.year, idx.quarter])
    elif frequency == "weekly":
        iso = idx.isocalendar()
        groups = pd.Series(idx, index=idx).groupby([iso.year, iso.week])
    else:
        raise ValueError(f"Unsupported rebalance frequency: {frequency}")

    return pd.DatetimeIndex([values.iloc[0] for _, values in groups])

def expand_rebalance_weights_to_daily(signal_weight_df, daily_index, rebalance_dates):
    signal = signal_weight_df.reindex(columns=ALL_ASSETS).fillna(0.0).copy()
    signal = normalize_rows(signal)
    signal_idx = pd.DatetimeIndex(signal.index).sort_values()

    daily = pd.DataFrame(index=daily_index, columns=ALL_ASSETS, dtype=float)

    for i, reb_date in enumerate(rebalance_dates):
        if i + 1 < len(rebalance_dates):
            period_idx = daily_index[(daily_index >= reb_date) & (daily_index < rebalance_dates[i + 1])]
        else:
            period_idx = daily_index[daily_index >= reb_date]

        prior_signals = signal_idx[signal_idx < reb_date]

        if len(prior_signals) == 0:
            signal_date = signal_idx[0]
        else:
            signal_date = prior_signals[-1]

        daily.loc[period_idx, ALL_ASSETS] = signal.loc[signal_date, ALL_ASSETS].values

    daily = daily.ffill().bfill()
    daily = normalize_rows(daily)

    return daily

def compute_turnover(daily_weights, rebalance_dates):
    turnover = pd.Series(0.0, index=daily_weights.index)
    prev_w = None

    for dt in rebalance_dates:
        if dt not in daily_weights.index:
            continue

        w = daily_weights.loc[dt, ALL_ASSETS]

        if prev_w is None:
            turnover.loc[dt] = w.drop(labels=[CASH_COL], errors="ignore").abs().sum()
        else:
            turnover.loc[dt] = (w - prev_w).abs().sum() / 2.0

        prev_w = w

    return turnover

def backtest_on_fixed_index(policy_name, signal_weights, etf_returns, evaluation_index):
    evaluation_index = pd.DatetimeIndex(evaluation_index).sort_values()

    returns = etf_returns.copy()
    returns[CASH_COL] = 0.0
    returns = returns.reindex(evaluation_index)

    if returns[ALL_ASSETS].isna().any().any():
        missing_rows = returns[returns[ALL_ASSETS].isna().any(axis=1)]
        raise ValueError(
            f"Return panel has missing rows for {policy_name}. "
            f"Example missing dates: {missing_rows.index[:5].tolist()}"
        )

    signal = signal_weights.copy()
    signal.index = pd.to_datetime(signal.index)
    signal = signal.sort_index()
    signal = signal.reindex(columns=ALL_ASSETS).fillna(0.0)

    signal_aligned = signal.reindex(evaluation_index).ffill().bfill()
    signal_aligned = normalize_rows(signal_aligned)

    rebalance_dates = get_rebalance_dates(evaluation_index, REBALANCE_FREQUENCY)

    daily_weights = expand_rebalance_weights_to_daily(
        signal_weight_df=signal_aligned,
        daily_index=evaluation_index,
        rebalance_dates=rebalance_dates,
    )

    gross_return = (daily_weights[ALL_ASSETS] * returns[ALL_ASSETS]).sum(axis=1)
    turnover = compute_turnover(daily_weights, rebalance_dates)
    transaction_cost = turnover * TRANSACTION_COST_RATE
    net_return = gross_return - transaction_cost

    equity = (1.0 + net_return).cumprod() * INITIAL_CAPITAL

    out = pd.DataFrame(index=evaluation_index)
    out.index.name = "date"
    out["policy_name"] = policy_name
    out["gross_return"] = gross_return
    out["turnover"] = turnover
    out["transaction_cost"] = transaction_cost
    out["net_return"] = net_return
    out["equity"] = equity
    out["drawdown"] = equity / equity.cummax() - 1.0
    out["is_rebalance_date"] = out.index.isin(rebalance_dates)

    return out, daily_weights

def performance_metrics(return_df):
    r = return_df["net_return"].astype(float).copy()
    equity = return_df["equity"].astype(float).copy()
    drawdown = return_df["drawdown"].astype(float).copy()

    n = len(r)
    if n == 0:
        return {}

    total_return = float(equity.iloc[-1] / equity.iloc[0] - 1.0) if equity.iloc[0] != 0 else np.nan
    annual_return = float((1.0 + total_return) ** (ANNUALIZATION_DAYS / max(n, 1)) - 1.0)

    annual_vol = float(r.std(ddof=1) * np.sqrt(ANNUALIZATION_DAYS)) if n > 1 else np.nan
    sharpe = annual_return / annual_vol if annual_vol and annual_vol > 0 else np.nan

    downside = r[r < 0]
    downside_vol = float(downside.std(ddof=1) * np.sqrt(ANNUALIZATION_DAYS)) if len(downside) > 1 else np.nan
    sortino = annual_return / downside_vol if downside_vol and downside_vol > 0 else np.nan

    max_drawdown = float(drawdown.min())
    calmar = annual_return / abs(max_drawdown) if max_drawdown < 0 else np.nan

    return {
        "n_days": int(n),
        "start_date": str(r.index.min().date()),
        "end_date": str(r.index.max().date()),
        "total_return": total_return,
        "annual_return": annual_return,
        "annual_volatility": annual_vol,
        "sharpe_ratio": sharpe,
        "sortino_ratio": sortino,
        "max_drawdown": max_drawdown,
        "calmar_ratio": calmar,
        "hit_rate": float((r > 0).mean()),
        "avg_daily_return": float(r.mean()),
        "avg_turnover": float(return_df["turnover"].mean()),
        "total_turnover": float(return_df["turnover"].sum()),
        "total_transaction_cost": float(return_df["transaction_cost"].sum()),
        "final_equity": float(equity.iloc[-1]),
    }


# ============================================================
# 5. Load ETF returns and probability inputs
# ============================================================

print("\n" + "=" * 80)
print("Loading ETF returns and Notebook 07 probabilities")
print("=" * 80)

etf_returns, etf_return_path = load_etf_return_panel()

print("ETF return panel:", etf_return_path)
print("ETF return shape:", etf_returns.shape)
print("ETF date range  :", etf_returns.index.min().date(), "to", etf_returns.index.max().date())

input_index_df = pd.read_csv(NOTEBOOK08_INPUT_INDEX)

p20_path = None
p60_path = None

for _, row in input_index_df.iterrows():
    target_col = row["target_col"]

    path = Path(row["probability_path_parquet"])
    if not path.exists():
        path = Path(row["probability_path_csv"])

    if "20d" in target_col:
        p20_path = path
    elif "60d" in target_col:
        p60_path = path

if p20_path is None or p60_path is None:
    raise ValueError("Could not locate both 20d and 60d probability files.")

p20_raw = read_table_auto(p20_path)
p60_raw = read_table_auto(p60_path)

for name, prob_df in [("p20_raw", p20_raw), ("p60_raw", p60_raw)]:
    missing = [c for c in proba_cols() if c not in prob_df.columns]
    if missing:
        raise ValueError(f"{name} missing probability columns: {missing}")

    if "fold_id" not in prob_df.columns:
        raise ValueError(f"{name} must include fold_id.")

    if "split" not in prob_df.columns:
        raise ValueError(f"{name} must include split.")

print("20d probability file:", p20_path, p20_raw.shape)
print("60d probability file:", p60_path, p60_raw.shape)

p20_test_latest = latest_fold_deduplicate(p20_raw, split_filter=["test"])
p60_test_latest = latest_fold_deduplicate(p60_raw, split_filter=["test"])

aligned_test_dates = (
    p20_test_latest.index
    .intersection(p60_test_latest.index)
    .intersection(etf_returns.index)
    .sort_values()
)

aligned_test_dates = aligned_test_dates[
    (aligned_test_dates >= pd.Timestamp(STRICT_START))
    & (aligned_test_dates <= pd.Timestamp(STRICT_END))
]

if len(aligned_test_dates) == 0:
    raise ValueError("No aligned strict-test dates found.")

print("Aligned strict-test dates:", len(aligned_test_dates))
print("Date range:", aligned_test_dates.min().date(), "to", aligned_test_dates.max().date())


# ============================================================
# 6. Define seven ablation variants
# ============================================================

def make_ablation_config(base, name, label, changes):
    cfg = copy.deepcopy(base)
    cfg["strategy_name"] = name
    cfg["ablation_label"] = label
    cfg["ablation_changes"] = changes

    for k, v in changes.items():
        cfg[k] = v

    return cfg

base_config = copy.deepcopy(BASE_UAMV_B_CONFIG)

ablation_configs = []

ablation_configs.append(
    make_ablation_config(
        base_config,
        name="ABL_FULL_AURORA10_UAMV_B",
        label="Full AURORA10-UAMV-B",
        changes={},
    )
)

ablation_configs.append(
    make_ablation_config(
        base_config,
        name="ABL_FIXED_RISK_AVERSION",
        label="Fixed risk aversion",
        changes={
            "uncertainty_risk_multiplier": 0.0,
            "bearish_risk_multiplier": 0.0,
        },
    )
)

ablation_configs.append(
    make_ablation_config(
        base_config,
        name="ABL_NO_ENTROPY_TERM",
        label="No entropy term",
        changes={
            "uncertainty_risk_multiplier": 0.0,
        },
    )
)

ablation_configs.append(
    make_ablation_config(
        base_config,
        name="ABL_NO_BEARISH_TERM",
        label="No bearish term",
        changes={
            "bearish_risk_multiplier": 0.0,
        },
    )
)

ablation_configs.append(
    make_ablation_config(
        base_config,
        name="ABL_NO_CASH",
        label="No cash allocation",
        changes={
            "max_cash_weight": 0.0,
            "min_cash_weight": 0.0,
        },
    )
)

ablation_configs.append(
    make_ablation_config(
        base_config,
        name="ABL_NO_TURNOVER_PENALTY",
        label="No turnover penalty",
        changes={
            "turnover_penalty": 0.0,
        },
    )
)

ablation_configs.append(
    make_ablation_config(
        base_config,
        name="ABL_NO_00881_SPECIAL_CAP",
        label="No 00881 special cap",
        changes={
            "max_00881_weight": base_config["max_etf_weight"],
        },
    )
)

print("\nAblation variants:")
for cfg in ablation_configs:
    print(cfg["strategy_name"], "|", cfg["ablation_label"], "| changes:", cfg["ablation_changes"])


# ============================================================
# 7. Run ablations
# ============================================================

print("\n" + "=" * 80)
print("Running AURORA10-UAMV-B component ablations")
print("=" * 80)

ablation_metric_rows = []
ablation_behavior_rows = []
ablation_return_frames = []
ablation_weight_frames = []
ablation_diag_frames = []

for cfg in ablation_configs:
    strategy_name = cfg["strategy_name"]
    ablation_label = cfg["ablation_label"]

    print("\nRunning:", strategy_name, "|", ablation_label)

    signal_w, diag_df = build_uamv_signal_weights(
        p20=p20_test_latest,
        p60=p60_test_latest,
        etf_returns=etf_returns,
        config=cfg,
        evaluation_dates=aligned_test_dates,
    )

    policy_name = f"AURORA10_{strategy_name}"

    ret_df, daily_w = backtest_on_fixed_index(
        policy_name=policy_name,
        signal_weights=signal_w,
        etf_returns=etf_returns,
        evaluation_index=aligned_test_dates,
    )

    metrics = performance_metrics(ret_df)

    metric_row = {
        "run_id": RUN_ID,
        "policy_name": policy_name,
        "strategy_name": strategy_name,
        "ablation_label": ablation_label,
        "period": "aligned_strict_test_ablation",
        "entropy_term": cfg["uncertainty_risk_multiplier"] > 0,
        "bearish_term": cfg["bearish_risk_multiplier"] > 0,
        "cash_allowed": cfg["max_cash_weight"] > 0,
        "turnover_penalty_active": cfg["turnover_penalty"] > 0,
        "special_00881_cap_active": cfg["max_00881_weight"] < cfg["max_etf_weight"],
        "alpha_20d": cfg["alpha_20d"],
        "alpha_60d": cfg["alpha_60d"],
        "base_risk_aversion": cfg["base_risk_aversion"],
        "uncertainty_risk_multiplier": cfg["uncertainty_risk_multiplier"],
        "bearish_risk_multiplier": cfg["bearish_risk_multiplier"],
        "turnover_penalty": cfg["turnover_penalty"],
        "max_etf_weight": cfg["max_etf_weight"],
        "max_00881_weight": cfg["max_00881_weight"],
        "max_cash_weight": cfg["max_cash_weight"],
        **metrics,
    }

    ablation_metric_rows.append(metric_row)

    cash = daily_w[CASH_COL]
    w_00881 = daily_w["00881"]

    behavior_row = {
        "run_id": RUN_ID,
        "policy_name": policy_name,
        "strategy_name": strategy_name,
        "ablation_label": ablation_label,
        "avg_cash_weight": float(cash.mean()),
        "median_cash_weight": float(cash.median()),
        "max_cash_weight_realized": float(cash.max()),
        "pct_days_at_cash_cap": float((cash >= cfg["max_cash_weight"] - 1e-8).mean()),
        "avg_00881_weight": float(w_00881.mean()),
        "median_00881_weight": float(w_00881.median()),
        "max_00881_weight_realized": float(w_00881.max()),
        "pct_days_at_00881_cap": float((w_00881 >= cfg["max_00881_weight"] - 1e-8).mean()),
        "avg_turnover": float(ret_df["turnover"].mean()),
        "total_turnover": float(ret_df["turnover"].sum()),
        "total_transaction_cost": float(ret_df["transaction_cost"].sum()),
        "num_rebalance_days": int(ret_df["is_rebalance_date"].sum()),
        "realized_annual_volatility": metrics["annual_volatility"],
    }

    ablation_behavior_rows.append(behavior_row)

    ret_out = ret_df.copy()
    ret_out["strategy_name"] = strategy_name
    ret_out["ablation_label"] = ablation_label
    ablation_return_frames.append(ret_out)

    w_out = daily_w.copy()
    w_out.insert(0, "ablation_label", ablation_label)
    w_out.insert(0, "strategy_name", strategy_name)
    w_out.insert(0, "policy_name", policy_name)
    ablation_weight_frames.append(w_out)

    diag_out = diag_df.copy()
    diag_out["policy_name"] = policy_name
    diag_out["strategy_name"] = strategy_name
    diag_out["ablation_label"] = ablation_label
    ablation_diag_frames.append(diag_out)

    ret_df.to_parquet(ABLATION_RETURN_DIR / f"returns_{safe_name(strategy_name)}.parquet")
    ret_df.to_csv(ABLATION_RETURN_DIR / f"returns_{safe_name(strategy_name)}.csv")

    daily_w.to_parquet(ABLATION_WEIGHT_DIR / f"weights_{safe_name(strategy_name)}.parquet")
    daily_w.to_csv(ABLATION_WEIGHT_DIR / f"weights_{safe_name(strategy_name)}.csv")

    diag_df.to_parquet(ABLATION_DIAG_DIR / f"diagnostics_{safe_name(strategy_name)}.parquet")
    diag_df.to_csv(ABLATION_DIAG_DIR / f"diagnostics_{safe_name(strategy_name)}.csv")


# ============================================================
# 8. Export ablation tables
# ============================================================

ablation_metrics_df = pd.DataFrame(ablation_metric_rows)
ablation_behavior_df = pd.DataFrame(ablation_behavior_rows)

ablation_returns_df = pd.concat(ablation_return_frames, axis=0)
ablation_weights_df = pd.concat(ablation_weight_frames, axis=0)
ablation_diag_df = pd.concat(ablation_diag_frames, axis=0)

ablation_metrics_df = ablation_metrics_df.sort_values(
    ["sharpe_ratio", "sortino_ratio", "max_drawdown"],
    ascending=[False, False, False],
)

ablation_metrics_path = ABLATION_TABLE_DIR / "table_S22_aurora_component_ablation_results.csv"
ablation_behavior_path = ABLATION_TABLE_DIR / "table_S23_cash_turnover_diagnostics_ablation.csv"

ablation_metrics_df.to_csv(ablation_metrics_path, index=False)
ablation_behavior_df.to_csv(ablation_behavior_path, index=False)

ablation_returns_df.to_parquet(ABLATION_RETURN_DIR / "all_ablation_returns.parquet")
ablation_returns_df.to_csv(ABLATION_RETURN_DIR / "all_ablation_returns.csv")

ablation_weights_df.to_parquet(ABLATION_WEIGHT_DIR / "all_ablation_weights.parquet")
ablation_weights_df.to_csv(ABLATION_WEIGHT_DIR / "all_ablation_weights.csv")

ablation_diag_df.to_parquet(ABLATION_DIAG_DIR / "all_ablation_diagnostics.parquet")
ablation_diag_df.to_csv(ABLATION_DIAG_DIR / "all_ablation_diagnostics.csv")

ablation_metrics_df.to_csv(
    TABLE_DIR / f"table_S22_aurora_component_ablation_results_{RUN_ID}.csv",
    index=False,
)

ablation_behavior_df.to_csv(
    TABLE_DIR / f"table_S23_cash_turnover_diagnostics_ablation_{RUN_ID}.csv",
    index=False,
)

print("\nAblation performance results:")
display(
    ablation_metrics_df[
        [
            "ablation_label",
            "entropy_term",
            "bearish_term",
            "cash_allowed",
            "turnover_penalty_active",
            "special_00881_cap_active",
            "total_return",
            "annual_return",
            "annual_volatility",
            "sharpe_ratio",
            "sortino_ratio",
            "max_drawdown",
            "calmar_ratio",
            "total_turnover",
            "total_transaction_cost",
        ]
    ].round(6)
)

print("\nAblation behavior diagnostics:")
display(
    ablation_behavior_df[
        [
            "ablation_label",
            "avg_cash_weight",
            "median_cash_weight",
            "max_cash_weight_realized",
            "pct_days_at_cash_cap",
            "avg_00881_weight",
            "max_00881_weight_realized",
            "pct_days_at_00881_cap",
            "avg_turnover",
            "total_turnover",
            "total_transaction_cost",
            "num_rebalance_days",
        ]
    ].round(6)
)

print("\nSaved ablation outputs:")
print("Metrics table :", ablation_metrics_path)
print("Behavior table:", ablation_behavior_path)
print("Returns dir   :", ABLATION_RETURN_DIR)
print("Weights dir   :", ABLATION_WEIGHT_DIR)
print("Diagnostics   :", ABLATION_DIAG_DIR)


# ============================================================
# 9. Save validation report
# ============================================================

validation_report = {
    "project_code": PROJECT_CODE,
    "notebook": "10B_AURORA_component_ablation_standalone_single_cell",
    "run_timestamp_utc": RUN_TIMESTAMP,
    "run_id": RUN_ID,
    "notebook07_run_id": NOTEBOOK07_RUN_ID,
    "etf_return_panel": str(etf_return_path),
    "probability_20d_path": str(p20_path),
    "probability_60d_path": str(p60_path),
    "strict_start": STRICT_START,
    "strict_end": STRICT_END,
    "aligned_test_n_days": int(len(aligned_test_dates)),
    "aligned_test_start": str(aligned_test_dates.min().date()),
    "aligned_test_end": str(aligned_test_dates.max().date()),
    "transaction_cost_rate": TRANSACTION_COST_RATE,
    "rebalance_frequency": REBALANCE_FREQUENCY,
    "has_scipy_optimizer": HAS_SCIPY,
    "base_config": BASE_UAMV_B_CONFIG,
    "ablation_configs": ablation_configs,
    "output_paths": {
        "run_root": str(RUN_ROOT),
        "ablation_metrics": str(ablation_metrics_path),
        "ablation_behavior": str(ablation_behavior_path),
        "returns": str(ABLATION_RETURN_DIR),
        "weights": str(ABLATION_WEIGHT_DIR),
        "diagnostics": str(ABLATION_DIAG_DIR),
    },
    "methodological_note": (
        "Each ablation changes one component relative to AURORA10-UAMV-B while preserving "
        "the same probability inputs, ETF universe, strict-test dates, optimizer, transaction-cost rate, "
        "and rebalance convention."
    ),
}

report_path = ABLATION_REPORT_DIR / "AURORA_10B_component_ablation_validation_report.json"
global_report_path = REPORT_DIR / f"AURORA_10B_component_ablation_validation_report_{RUN_ID}.json"

save_json(report_path, validation_report)
save_json(global_report_path, validation_report)

print("\nValidation report saved:")
print(report_path)
print(global_report_path)

print("\n" + "=" * 80)
print("AURORA COMPONENT ABLATION COMPLETE")
print("=" * 80)
print("Run ID:", RUN_ID)
print("Run root:", RUN_ROOT)
print("=" * 80)

Mounted at /content/drive
AURORA-TWETF standalone component ablation
RUN_ID: 20260711_150104
RUN_ROOT: /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/aurora_component_ablation/run_20260711_150104
Notebook 07 input index: /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/purged_walk_forward_models/run_20260624_031817/NOTEBOOK08_OR_ALLOCATION_INPUT_INDEX_PURGED_WF.csv

Loading ETF returns and Notebook 07 probabilities
ETF return panel: /content/drive/MyDrive/AURORA_TWETF/data/panels/AURORA_etf_return_panel.parquet
ETF return shape: (1426, 4)
ETF date range  : 2021-01-01 to 2026-06-23
20d probability file: /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/purged_walk_forward_models/run_20260624_031817/probabilities/selected_probabilities_TAIEX_regime_fixed_20d_E1_validation_weighted_probability_ensemble.parquet (1015, 11)
60d probability file: /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/purged_walk_forward_models/run_20260624_031817/probabilities/selec

,ablation_label,entropy_term,bearish_term,cash_allowed,turnover_penalty_active,special_00881_cap_active,total_return,annual_return,annual_volatility,sharpe_ratio,sortino_ratio,max_drawdown,calmar_ratio,total_turnover,total_transaction_cost
4,No cash allocation,True,True,False,True,True,0.651613,0.486417,0.245592,1.980586,2.549032,-0.275074,1.768316,1.329678,0.001330
3,No bearish term,True,False,True,True,True,0.265292,0.204280,0.104237,1.959765,2.616862,-0.116909,1.747342,1.617266,0.001617
0,Full AURORA10-UAMV-B,True,True,True,True,True,0.263120,0.202647,0.103730,1.953601,2.602030,-0.116788,1.735174,1.617999,0.001618
6,No 00881 special cap,True,True,True,True,False,0.263120,0.202647,0.103730,1.953601,2.602030,-0.116788,1.735174,1.617999,0.001618
5,No turnover penalty,True,True,True,False,True,0.228099,0.176228,0.097556,1.806435,2.296497,-0.118361,1.488900,1.004484,0.001004
2,No entropy term,False,True,True,True,True,0.241942,0.186689,0.121529,1.536175,1.933286,-0.155935,1.197226,1.614883,0.001615
1,Fixed risk aversion,False,False,True,True,True,0.232850,0.179821,0.135205,1.329992,1.610574,-0.181484,0.990839,1.640440,0.001640



Ablation behavior diagnostics:


,ablation_label,avg_cash_weight,median_cash_weight,max_cash_weight_realized,pct_days_at_cash_cap,avg_00881_weight,max_00881_weight_realized,pct_days_at_00881_cap,avg_turnover,total_turnover,total_transaction_cost,num_rebalance_days
0,Full AURORA10-UAMV-B,0.556502,0.600000,0.6,0.874608,0.012357,0.141314,0.0,0.005072,1.617999,0.001618,17
1,Fixed risk aversion,0.480852,0.586663,0.6,0.332288,0.034051,0.147407,0.0,0.005142,1.640440,0.001640,17
2,No entropy term,0.508162,0.600000,0.6,0.567398,0.026025,0.146632,0.0,0.005062,1.614883,0.001615,17
3,No bearish term,0.554146,0.600000,0.6,0.874608,0.012825,0.142056,0.0,0.005070,1.617266,0.001617,17
4,No cash allocation,0.000000,0.000000,0.0,1.000000,0.069150,0.149234,0.0,0.004168,1.329678,0.001330,17
5,No turnover penalty,0.600000,0.600000,0.6,1.000000,0.011129,0.088695,0.0,0.003149,1.004484,0.001004,17
6,No 00881 special cap,0.556502,0.600000,0.6,0.874608,0.012357,0.141314,0.0,0.005072,1.617999,0.001618,17



Saved ablation outputs:
Metrics table : /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/aurora_component_ablation/run_20260711_150104/tables/table_S22_aurora_component_ablation_results.csv
Behavior table: /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/aurora_component_ablation/run_20260711_150104/tables/table_S23_cash_turnover_diagnostics_ablation.csv
Returns dir   : /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/aurora_component_ablation/run_20260711_150104/returns
Weights dir   : /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/aurora_component_ablation/run_20260711_150104/weights
Diagnostics   : /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/aurora_component_ablation/run_20260711_150104/diagnostics

Validation report saved:
/content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/aurora_component_ablation/run_20260711_150104/reports/AURORA_10B_component_ablation_validation_report.json
/content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_